# Проект: Яндекс.Книги. Проверка гипотезы в Python. Анализ результатов A/B-тестирования

- Автор: Мельников Даниил
- Дата: 28.04.2026

## Содержание  
[Часть 1. Проверка гипотезы в Python и составление аналитической записки](#часть-1-проверка-гипотезы-в-python-и-составление-аналитической-записки)  
&nbsp;&nbsp;[1. Цели и задачи проекта](#цели-и-задачи-проекта)  
&nbsp;&nbsp;[2. Описание данных](#описание-данных)  
&nbsp;&nbsp;[3. Загрузка данных и знакомство с ними](#загрузка-данных-и-знакомство-с-ними)  
&nbsp;&nbsp;[4. Проверка гипотезы в Python](#проверка-гипотезы-в-Python)  
&nbsp;&nbsp;[5. Аналитическая записка](#аналитическая-записка)  
[Часть 2. Анализ результатов A/B-тестирования](#анализ-результатов-A/B-тестирования)  
&nbsp;&nbsp;[1. Цели исследования](#цели-исследования)  
&nbsp;&nbsp;[2. Загрузка данных, оценка их целостности](#загрузка-данных,-оценка-их-целостности)  
&nbsp;&nbsp;[3. Оценка результатов A/B-тестирования](#оценка-результатов-A/B-тестирования)

# Часть 1. Проверка гипотезы в Python и составление аналитической записки

## Цели и задачи проекта

**Цель проекта** - проверить гипотезу, согласно которой пользователи с большим LTV являются более лояльными и проводят больше времени в приложении.  
Для достижения цели проекта были опеределены следующие **задачи**:
1. Сформулировать нулевую и альтернативную гипотезы для проведения стат теста
2. Провести стат тест, интерпретировать его результаты

## Описание данных

Таблицы этого проекта содержат данные о чтении и прослушивании контента в сервисе Яндекс Книги, которые включают информацию о пользователях, платформах, времени, длительности сессий и типах контента. Данные представлены за период с 1 сентября по 11 декабря 2024 года. В вашем распоряжении будет несколько таблиц.  

Таблица `bookmate.audition` содержит данные об активности пользователей и состоит из следующих полей:  
`audition_id` — уникальный идентификатор сессии чтения или прослушивания;  
`puid` — идентификатор пользователя;  
`usage_platform_ru` — название платформы, с помощью которой пользователь слушал контент;  
`msk_business_dt_str` — дата события в формате строки (московское время);  
`app_version` — версия приложения, которая использовалась для чтения или прослушивания;  
`adult_content_flg` — был ли это контент для взрослых: True или False;  
`hours` — длительность чтения или прослушивания в часах;  
`hours_sessions_long` — продолжительность длинных сессий чтения или прослушивания в часах;  
`kids_content_flg` — был ли это детский контент: True или False;  
`main_content_id` — идентификатор основного контента;  
`usage_geo_id` — идентификатор географического местоположения.  

Таблица `bookmate.content` содержит данные о контенте и состоит из следующих полей:  
`main_content_id` — идентификатор основного контента;  
`main_author_id` — идентификатор основного автора контента;  
`main_content_type` — тип контента;  
`main_content_name`— название контента;  
`main_content_duration_hours` — длительность контента в часах;  
`published_topic_title_list` — список жанров контента.  

Таблица `bookmate.author` содержит данные об авторах контента и состоит из следующих полей:  
`main_author_id` — идентификатор основного автора контента;  
`main_author_name` — имя основного автора контента.  

Таблица `bookmate.geo` содержит данные о местоположении и состоит из следующих полей:  
`usage_geo_id` — идентификатор географического положения;  
`usage_geo_id_name` — город или регион географического положения;  
`usage_country_name` — страна географического положения.

## Загрузка данных и знакомство с ними

In [33]:
import pandas as pd
from scipy.stats import mannwhitneyu
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

In [2]:
yandex_knigi_data = pd.read_csv('/datasets/yandex_knigi_data.csv').drop(columns=['Unnamed: 0'])

yandex_knigi_data.head()

,city,puid,hours
0,Москва,9668,26.167776
1,Москва,16598,82.111217
2,Москва,80401,4.656906
3,Москва,140205,1.840556
4,Москва,248755,151.326434


In [20]:
# Проверяем наличие дубликатов
explicit_duplicates = yandex_knigi_data.duplicated().sum()
implicit_duplicates = yandex_knigi_data.duplicated(subset=['puid', 'hours']).sum()

explicit_duplicates, implicit_duplicates

(np.int64(0), np.int64(0))

Дубликаты отстутсвуют.

In [21]:
# Оцениваем размеры выборок
msk = yandex_knigi_data[yandex_knigi_data['city'] == 'Москва']['puid'].nunique()
spb = yandex_knigi_data[yandex_knigi_data['city'] == 'Санкт-Петербург']['puid'].nunique()

print(f'Размер группы А: {msk}')
print(f'Размер группы В: {spb}')

Размер группы А: 6234
Размер группы В: 2550


Размер группы B сильно меньше, чем размер группы А.

Выбираем тест Манна-Уитни, т.к. данные о часах не имеют нормального распределения.

## Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [7]:
msk = yandex_knigi_data[yandex_knigi_data['city'] == 'Москва']['hours']
spb = yandex_knigi_data[yandex_knigi_data['city'] == 'Санкт-Петербург']['hours']


stat_mw, p_value_mw = mannwhitneyu(
    msk,
    spb,
    alternative='greater'
)

print(f'Значение pvalue = {p_value_mw}')

alpha = 0.05

if p_value_mw > alpha:
    print(f'Нулевая гипотеза верна! Статистически значимых различий нет!')
else:
    print(f'Отвергаем нулевую гипотезу! Альтернативная гипотеза верна!')

Значение pvalue = 0.9109420539159508
Нулевая гипотеза верна! Статистически значимых различий нет!


## Аналитическая записка

Для проверки гипотезы был выбран тест Манна-Уитни, т.к. мы рассматриваем кол-во проведенных часов в приложении. Использование параметрического теста могло бы дать некорректные результаты, поэтому был выбран непараметрический - тест Манна-Уитни.  
Уровень значимости задан стандартный - 5%. Значение pvalue = 0.91. Делаем вывод, что средняя активность пользователей в Санкт-Петербурге не больше, чем в Москве, соответственно, альтернативная гипотеза неверна. Причиной можно считать тот факт, что выборка по Санкт-Петербургу сильно меньше, чем в Москве, а данные имеют высокий разброс.

----

# Часть 2. Анализ результатов A/B-тестирования

В данной части проекта нужно проанализировать другие данные.  
Контекст: к вам обратились представители интернет-магазина BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни. У него есть своя целевая аудитория, даже появились хиты продаж: эспандер со счётчиком и напоминанием, так и подстольный велотренажёр с Bluetooth.

В будущем компания хочет расширить ассортимент товаров. Но перед этим нужно решить одну проблему. Интерфейс онлайн-магазина слишком сложен для пользователей — об этом говорят отзывы.

Чтобы привлечь новых клиентов и увеличить число продаж, владельцы магазина разработали новую версию сайта и протестировали его на части пользователей. По задумке, это решение доказуемо повысит количество пользователей, которые совершат покупку.

Задача — провести оценку результатов A/B-теста. 

## Цели исследования

Проверить действительно ли новый дизайн сайта увеличивает конверсию в покупку.

## Загрузка данных, оценка их целостности


In [8]:
participants = pd.read_csv('/datasets/ab_test_participants.csv')
events = pd.read_csv('/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

In [9]:
participants.info()
participants.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


Присутствуют данные для другого А/В-теста.

In [10]:
events.info()
events.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB


,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


Пропуски в столбце details допустимы и не имеют ключевого значения для проведения A/B-теста.

In [23]:
# Оцениваем размеры выборок
test_participants = participants[participants['ab_test'] == 'interface_eu_test']

group_a = test_participants[test_participants['group'] == 'A']['user_id'].nunique()
group_b = test_participants[test_participants['group'] == 'B']['user_id'].nunique()

display(group_a, group_b)

perc_diff = 100 * (group_a - group_b) / group_a
perc_diff

5383

5467

-1.5604681404421326

Разница в размерах выборок составляет примерно 1.6%, что допустимо при проведении A/B-тестирования.

In [24]:
# Находим пересечения в группах с другим А/В-тестом
interface_eu_test = participants[participants['ab_test'] == 'interface_eu_test']['user_id']
recommender_system_test = participants[participants['ab_test'] == 'recommender_system_test']['user_id']

intersection = list(set(interface_eu_test) & set(recommender_system_test))

clean_test_participants = participants[~participants['user_id'].isin(intersection)]

In [26]:
# Оставляем пользователей из нужного А/В-теста
test_users = clean_test_participants['user_id'].unique()

test_events = events[events['user_id'].isin(test_users)]

In [27]:
# Оставляем события, совершенный в течение 7 дней с момента регистрации
reg_dates = events[events['event_name'] == 'registration'][['user_id', 'event_dt']]
reg_dates = reg_dates.rename(columns={'event_dt': 'reg_date'})

test_events_with_reg = test_events.merge(reg_dates, on='user_id', how='inner')

test_events_with_reg['lifetime_days'] = (test_events_with_reg['event_dt'] - test_events_with_reg['reg_date']).dt.days

events_7_days = test_events_with_reg[test_events_with_reg['lifetime_days'] <= 6]

Оцените достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [30]:
# Оцениваем достаточность выборки
baseline = 0.30
effect_abs = 0.03  
alpha = 0.05
power = 0.80

p2 = baseline + effect_abs

p_pooled = (baseline + p2) / 2
effect_size = effect_abs / np.sqrt(p_pooled * (1 - p_pooled))

power_analysis = NormalIndPower()
sample_size = power_analysis.solve_power(
    effect_size=effect_size,
    power=power,
    alpha=alpha,
    ratio=1
)

print(f"Необходимый размер выборки для каждой группы: {int(np.ceil(sample_size))}")

Необходимый размер выборки для каждой группы: 3764


In [31]:
# Рассчитываем общее кол-во пользователей и в каждой группе
purchase_users = events_7_days[events_7_days['event_name'] == 'purchase']['user_id'].unique()

purchases_a = test_participants[(test_participants['group'] == 'A') & (test_participants['user_id'].isin(purchase_users))]['user_id'].nunique()
purchases_b = test_participants[(test_participants['group'] == 'B') & (test_participants['user_id'].isin(purchase_users))]['user_id'].nunique()

In [32]:
print('Группа А:')
print(f'Размер группы: {group_a} Кол-во покупок: {purchases_a} Конверсия: {purchases_a / group_a * 100}')
print('Группа B:')
print(f'Размер группы: {group_b} Кол-во покупок: {purchases_b} Конверсия: {purchases_b / group_b * 100}')

Группа А:
Размер группы: 5383 Кол-во покупок: 1377 Конверсия: 25.580531302247817
Группа B:
Размер группы: 5467 Кол-во покупок: 1480 Конверсия: 27.07152002926651


Увеличение конверсии не соответсвует ожидаемому (на 3 процентных пункта). Необходимо проверить имеет ли статистическую значимость наблюдаемое изменение конверсии.

## Оценка результатов A/B-тестирования

**Нулевая гипотеза**: новый дизайн сайта не придал статистически значимых изменений с точки зрения уровня конверсии.  
**Альтернативная гипотеза**: новый дизайн сайта повысил уровень конверсии, и данное изменение имеет статистическую значимость.

In [19]:
alpha = 0.05

stat_ztest, p_value_ztest = proportions_ztest(
    [purchases_a, purchases_b],
    [group_a, group_b],
    alternative='smaller')

print(f'Значение pvalue = {p_value_ztest}')

if p_value_ztest > alpha:
    print(f'Нулевая гипотеза верна! Новый дизайн сайта не улучшил показатели конверсии!')
else:
    print(f'Альтернативная гипотеза верна! Новый дизайн сайта улучшил конверсию в покупку!')

Значение pvalue = 0.038945583501089065
Альтернативная гипотеза верна! Новый дизайн сайта улучшил конверсию в покупку!


Согласно проведенному z-тесту пропорций, новый дизайн сайта повысил урвоень конверсии, и данное повышение имеет статистическую значимость. Следовательно, новый дизайн сайта интернет-магазина рекомендуется к внедрению в качестве полноценной замены текущего интерфейса.